# 01: Environment check

Confirms the code, PyTorch, the device and the runs folder work in the current runtime.
Run locally, or in Colab with a GPU runtime (Runtime > Change runtime type > T4 GPU).

In [ ]:
# Colab setup: fetch this repository's latest code from GitHub. Does nothing locally.
# Same cell in every notebook. After pushing new code: Runtime > Restart session, then run all cells.
import sys

REPO = "dgelbier23/dental-3d-vae"

if "google.colab" in sys.modules:
    import subprocess
    from pathlib import Path
    from google.colab import userdata

    try:  # A token is only needed while the repository is private.
        token = userdata.get("GITHUB_TOKEN")
    except Exception:  # No such secret, or this notebook may not read it.
        token = ""
    login = f"x-access-token:{token}@" if token else ""
    repo_dir = Path("/content") / REPO.split("/")[1]
    subprocess.run(["rm", "-rf", str(repo_dir)], check=True)
    result = subprocess.run(["git", "clone", "--depth", "1", f"https://{login}github.com/{REPO}.git", str(repo_dir)],
                            capture_output=True, text=True)
    if result.returncode:
        error = result.stderr.replace(token, "***") if token else result.stderr
        raise RuntimeError(f"{error} If the repository is private, add a GITHUB_TOKEN Colab secret (see the README).")
    if token:  # git clone saves the URL it was given, so remove the token from the clone's settings.
        subprocess.run(["git", "-C", str(repo_dir), "remote", "set-url", "origin",
                        f"https://github.com/{REPO}.git"], check=True)
    sys.path.insert(0, str(repo_dir / "src"))

In [ ]:
import torch

from dental_3d_vae.runtime.env import describe_environment, get_runs_dir
from dental_3d_vae.ml.model import SimpleClassifier
from dental_3d_vae.runtime.device import get_device

for key, value in describe_environment().items():
    print(f"{key:>10}: {value}")

device = get_device()
outputs = SimpleClassifier().to(device)(torch.randn(4, 20, device=device))
print(f"{'model':>10}: output {tuple(outputs.shape)} on {outputs.device}")
print(f"{'runs dir':>10}: {get_runs_dir()}")